# Build with Sarvam AI in the Vercel AI SDK

The **[Vercel AI SDK](https://ai-sdk.dev)** is a TypeScript toolkit for building AI-powered applications: one set of functions (`generateText`, `streamText`, `generateObject`, and more) that work the same way across different model providers.

**[sarvam-ai-sdk](https://github.com/sarvamai/sarvam-ai-sdk)** is a community provider for the Vercel AI SDK that wraps Sarvam's **chat completion**, **text-to-speech**, **speech-to-text**, **translation**, **transliteration**, and **language identification** APIs behind the AI SDK's standard functions: `generateText`, `generateObject`, `generateSpeech`, and `transcribe`.

If you already build with the AI SDK, this lets you swap in Sarvam models without changing how you call `generateText` or stream responses — tool calling and structured output work the same way as with any other AI SDK provider.

> **This notebook is different from the others in `integrations/`:** the Vercel AI SDK is a TypeScript/Node.js library, not Python. The Python kernel here is only the orchestrator — each code cell uses `%%writefile` to save a `.ts` file and then runs it with `!npx tsx`, so you need **Node.js** installed, but nothing beyond a normal Jupyter/Python environment otherwise.

## Compatibility

`sarvam-ai-sdk` versions track a specific major version of `ai`. Install matching versions or you will hit type and runtime mismatches.

| Sarvam AI SDK version | Vercel AI SDK version |
|---|---|
| `0.5.x` (beta) | `8.x.x` (beta) |
| `0.4.x` (current) | `7.x.x` (current) |
| `0.3.x` | `6.x.x` |
| `0.2.x` | `5.x.x` |
| `0.1.x` | `4.x.x` |

This notebook pins `ai@7`, matching `sarvam-ai-sdk`'s current (`0.4.x`) release.

## Prerequisites

- **Node.js 18+** and **npm** available on `PATH` (check with `!node --version` below).
- A [Sarvam AI](https://dashboard.sarvam.ai) API key.

In [ ]:
# Confirm Node.js and npm are available before going any further.
!node --version
!npm --version

## 1. Set up a scratch Node.js project

To keep this self-contained, we create a small project folder alongside the notebook, `sarvam_vercel_ai_sdk_demo/`, and install:

- `sarvam-ai-sdk` — the community Sarvam provider
- `ai@7` — the Vercel AI SDK, pinned per the compatibility table above
- `zod` — schema validation, used later for tool calling and structured output
- `dotenv` — loads `.env` into `process.env`
- `tsx` — runs TypeScript files directly with `npx tsx file.ts`, no separate build step

`%cd` changes the notebook kernel's working directory (not just a subshell's), so every cell after this one runs from inside the project folder.

We also set `"type": "module"` in `package.json` — the examples below use top-level `await`, which `tsx` only supports when the project is treated as an ES module.

In [ ]:
!mkdir -p sarvam_vercel_ai_sdk_demo
%cd sarvam_vercel_ai_sdk_demo
!npm init -y >/dev/null
!npm pkg set type=module
!npm install --silent sarvam-ai-sdk ai@7 zod dotenv tsx

## 2. Set your API key

Get an API key from [dashboard.sarvam.ai](https://dashboard.sarvam.ai). Never commit real API keys to version control — the cell below writes a `.env.example` template; copy it to `.env` and fill in your real key locally:

```bash
cp .env.example .env
```

`sarvam` (from `sarvam-ai-sdk`) reads `SARVAM_API_KEY` from the environment automatically, so no explicit client setup is needed — each `.ts` file below just needs `import "dotenv/config"` at the top to load `.env` into `process.env`.

In [ ]:
%%writefile .env.example
# Copy this file to `.env` and fill in your real credentials.
# Do not commit `.env` to version control.

SARVAM_API_KEY=your_sarvam_api_key

## 3. Generate text

`generateText` is the AI SDK's core function for chat/completions. Point `model` at `sarvam(...)` and everything else — prompts, streaming, tool calling — works the same as with any other AI SDK provider.

In [ ]:
%%writefile generate-text.ts
import "dotenv/config";
import { sarvam } from "sarvam-ai-sdk";
import { generateText } from "ai";

try {
  const { text } = await generateText({
    model: sarvam("sarvam-30b"),
    prompt: "Translate this to malayalam: 'Keep cooking, guys'",
  });

  console.log(text); // e.g. "പാചകം തുടരൂ, സുഹൃത്തുക്കളേ"
} catch (error) {
  console.error("generateText failed:", error);
}

In [ ]:
!npx tsx generate-text.ts

### Tip: handling truncated responses

If longer completions come back truncated mid-sentence, the model is hitting its output token limit. Raise the limit and check the finish reason to confirm when a response was truncated. Through the AI SDK this is `maxOutputTokens` / `finishReason`; calling Sarvam directly (Python or REST) it's the underlying `max_tokens` / `finish_reason`.

In [ ]:
%%writefile generate-text-max-tokens.ts
import "dotenv/config";
import { sarvam } from "sarvam-ai-sdk";
import { generateText } from "ai";

const result = await generateText({
  model: sarvam("sarvam-30b"),
  prompt: "Write a detailed explanation of the water cycle.",
  maxOutputTokens: 4096,
});

console.log(result.text.slice(0, 200) + "...");

if (result.finishReason === "length") {
  console.warn("Response was truncated; consider raising maxOutputTokens further.");
} else {
  console.log(`finishReason: ${result.finishReason} (not truncated)`);
}

In [ ]:
!npx tsx generate-text-max-tokens.ts

The same check in **Python**, calling Sarvam directly with the official SDK:

```python
from sarvamai import SarvamAI

client = SarvamAI(api_subscription_key="YOUR_SARVAM_API_KEY")

response = client.chat.completions(
    model="sarvam-30b",
    messages=[{"role": "user", "content": "Write a detailed explanation of..."}],
    max_tokens=4096,
)

if response.choices[0].finish_reason == "length":
    print("Response was truncated; consider raising max_tokens further.")
```

And over raw **cURL**:

```bash
curl -X POST https://api.sarvam.ai/v1/chat/completions \
  -H "api-subscription-key: $SARVAM_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
    "model": "sarvam-30b",
    "messages": [{"role": "user", "content": "Write a detailed explanation of..."}],
    "max_tokens": 4096
  }'
```

Check `choices[0].finish_reason` in the response; `"length"` means the output was truncated.

## Text-to-speech

Use `generateSpeech` with `sarvam.speech(model, targetLanguage)`. This cell writes the generated audio to `output.wav`, which we'll reuse in the next section to demonstrate speech-to-text on the exact same clip.

In [ ]:
%%writefile text-to-speech.ts
import "dotenv/config";
import { sarvam } from "sarvam-ai-sdk";
import { generateSpeech } from "ai";
import { writeFile } from "fs/promises";

try {
  const { audio } = await generateSpeech({
    model: sarvam.speech("bulbul:v3", "ml-IN"),
    text: "പാചകം തുടരൂ, സുഹൃത്തുക്കളേ",
  });

  const audioBuffer = Buffer.from(audio.base64, "base64");
  await writeFile("./output.wav", audioBuffer);
  console.log("Wrote output.wav");
} catch (error) {
  console.error("generateSpeech failed:", error);
}

In [ ]:
!npx tsx text-to-speech.ts

See [Text to Speech](https://docs.sarvam.ai/api/api-guides-tutorials/text-to-speech/overview) for supported languages, speakers, and audio options available on the underlying API.

## Speech-to-text

Use `transcribe` with `sarvam.transcription(model, language)`. We point this at the `output.wav` file generated above, so you can see the full text → speech → text round trip.

In [ ]:
%%writefile speech-to-text.ts
import "dotenv/config";
import { sarvam } from "sarvam-ai-sdk";
import { transcribe } from "ai";
import { readFile } from "fs/promises";

try {
  const { text } = await transcribe({
    model: sarvam.transcription("saaras:v3", "ml-IN"),
    audio: await readFile("./output.wav"),
  });

  console.log(text); // transcript of the Malayalam clip generated in the TTS step
} catch (error) {
  console.error("transcribe failed:", error);
}

In [ ]:
!npx tsx speech-to-text.ts

See [Saaras](https://docs.sarvam.ai/api/getting-started/models/saaras) and [Saarika (Legacy)](https://docs.sarvam.ai/api/getting-started/models/saarika) for model details.

## Translation

In [ ]:
%%writefile translate.ts
import "dotenv/config";
import { sarvam } from "sarvam-ai-sdk";
import { generateText } from "ai";

try {
  const result = await generateText({
    model: sarvam.translation("mayura:v1", {
      from: "ml-IN",
      to: "en-IN",
    }),
    prompt: "ഇതൊക്കെ ശ്രദ്ധിക്കണ്ടേ അംബാനെ?",
  });

  console.log(result.text); // e.g. "Shouldn't we be careful about this, Ambane?"
} catch (error) {
  console.error("Translation failed:", error);
}

In [ ]:
!npx tsx translate.ts

Only `prompt` and `role: user` messages are translated; `system` and `assistant` messages pass through unchanged.

## Transliteration

In [ ]:
%%writefile transliterate.ts
import "dotenv/config";
import { sarvam } from "sarvam-ai-sdk";
import { generateText } from "ai";

try {
  const result = await generateText({
    model: sarvam.transliterate({
      to: "ml-IN",
      from: "en-IN", // optional
    }),
    prompt: "eda mone, happy alle?",
  });

  console.log(result.text); // e.g. "എടാ മോനെ, ഹാപ്പി അല്ലേ?"
} catch (error) {
  console.error("Transliteration failed:", error);
}

In [ ]:
!npx tsx transliterate.ts

## Language identification

In [ ]:
%%writefile language-id.ts
import "dotenv/config";
import { sarvam } from "sarvam-ai-sdk";
import { generateText } from "ai";

try {
  const result = await generateText({
    model: sarvam.languageIdentification(),
    prompt: "ബുദ്ധിയാണ് സാറേ ഇവൻ്റെ മെയിൻ",
  });

  console.log(result.text); // e.g. "ml-IN"
} catch (error) {
  console.error("Language identification failed:", error);
}

In [ ]:
!npx tsx language-id.ts

## Tool calling

Sarvam chat models support the AI SDK's standard `tool()` interface.

In [ ]:
%%writefile tool-calling.ts
import "dotenv/config";
import { z } from "zod";
import { generateText, tool } from "ai";
import { sarvam } from "sarvam-ai-sdk";

try {
  const result = await generateText({
    model: sarvam("sarvam-30b"),
    tools: {
      weather: tool({
        description: "Get the weather in a location",
        inputSchema: z.object({
          location: z.string(),
        }),
        execute: async ({ location }) => ({
          location,
          temperature: 72 + Math.floor(Math.random() * 21) - 10,
        }),
      }),
    },
    instructions: "You are a helpful AI",
    prompt: "കൊച്ചിയിലെ കാലാവസ്ഥ എന്താണ്?", // "What's the weather in Kochi?"
  });

  console.log(JSON.stringify(result.toolResults, null, 2));
} catch (error) {
  console.error("Tool call failed:", error);
}

In [ ]:
!npx tsx tool-calling.ts

## Structured output

Prefer `generateText` with `Output.object`; `generateObject` still works but is deprecated in the AI SDK.

In [ ]:
%%writefile structured-output.ts
import "dotenv/config";
import { z } from "zod";
import { sarvam } from "sarvam-ai-sdk";
import { generateText, Output } from "ai";

try {
  const { output } = await generateText({
    model: sarvam("sarvam-105b"),
    output: Output.object({
      name: "Recipe",
      description: "A recipe with a name, ingredients and steps",
      schema: z.object({
        recipe: z.object({
          name: z.string(),
          ingredients: z.array(z.string()),
          steps: z.array(z.string()),
        }),
      }),
    }),
    prompt: "Generate a South Indian recipe, in Malayalam",
  });

  console.log(JSON.stringify(output, null, 2));
} catch (error) {
  console.error("Structured output generation failed:", error);
}

In [ ]:
!npx tsx structured-output.ts

## All APIs at a glance

```ts
import { sarvam } from "sarvam-ai-sdk";

// Chat completion
sarvam("sarvam-105b");
sarvam.languageModel("sarvam-30b");

// Transliteration
sarvam.transliterate({ to: "ml-IN", from: "en-IN" });

// Translation
sarvam.translation("mayura:v1", { from: "en-IN", to: "ml-IN" });

// Language identification
sarvam.languageIdentification();

// Text-to-speech
sarvam.speech("bulbul:v3", "ml-IN");

// Speech-to-text
sarvam.transcription("saaras:v3");
```

| Function | AI SDK entry point | Underlying Sarvam capability |
|---|---|---|
| `sarvam("sarvam-105b")` / `sarvam.languageModel(...)` | `generateText`, `generateObject`, `streamText` | [Chat completion](https://docs.sarvam.ai/api/api-guides-tutorials/chat-completion/overview) |
| `sarvam.translation(model, { from, to })` | `generateText` | [Translation](https://docs.sarvam.ai/api/api-guides-tutorials/text-processing/translation) |
| `sarvam.transliterate({ from, to })` | `generateText` | Transliteration |
| `sarvam.languageIdentification()` | `generateText` | [Language identification](https://docs.sarvam.ai/api/api-guides-tutorials/text-processing/language-detection) |
| `sarvam.speech(model, targetLanguage)` | `generateSpeech` | [Text to Speech](https://docs.sarvam.ai/api/api-guides-tutorials/text-to-speech/overview) |
| `sarvam.transcription(model, language)` | `transcribe` | [Speech to Text](https://docs.sarvam.ai/api/api-guides-tutorials/speech-to-text/overview) |

## Troubleshooting

- **Type or runtime errors after install** — Your `sarvam-ai-sdk` and `ai` versions are mismatched. Check the compatibility table above and pin `ai` to the major version your `sarvam-ai-sdk` version expects (for example `ai@6` with `sarvam-ai-sdk@0.3.x`).
- **401/403 from Sarvam** — Confirm `SARVAM_API_KEY` is set in the environment the process actually reads (`.env` loaded, no stray whitespace in the key). Sarvam's REST API returns `403` (not `401`) for auth failures.
- **Translation/transliteration/LID returns the input unchanged** — These wrappers only act on `prompt` and `role: user` messages; `system` and `assistant` messages are passed through as-is by design.
- **`generateObject` deprecation warnings** — Migrate to `generateText` with `Output.object(...)` as shown above; `generateObject` still works but is deprecated upstream in the AI SDK.
- **Responses get cut off partway through** — Output length is capped by default (`maxOutputTokens` in the AI SDK, `max_tokens` when calling Sarvam directly from Python or REST). Raise the limit and check `finishReason` / `finish_reason` equals `"length"` to confirm truncation — see the [Generate text](#3-generate-text) section above.

## Additional resources

- [sarvam-ai-sdk on GitHub](https://github.com/sarvamai/sarvam-ai-sdk) (source, issues, changelog)
- [sarvam-ai-sdk on npm](https://npmjs.com/sarvam-ai-sdk)
- [Sarvam provider on the AI SDK docs](https://v7.ai-sdk.dev/providers/community-providers/sarvam)
- [Chat completion overview](https://docs.sarvam.ai/api/api-guides-tutorials/chat-completion/overview)
- [Text to Speech overview](https://docs.sarvam.ai/api/api-guides-tutorials/text-to-speech/overview)
- [Speech to Text overview](https://docs.sarvam.ai/api/api-guides-tutorials/speech-to-text/overview)
- [Build workflows with Sarvam AI in n8n](./n8n_workflow_automation.ipynb) — same Sarvam capabilities without writing code

## Need help?

- Sarvam Support: developer@sarvam.ai
- Community: [Join the Discord Community](https://discord.com/invite/5rAsykttcs)